# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [7]:
import sys, os, textwrap

sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
from src.configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig, DatasetConfig, DenoisingConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE
from src.denoiser import Denoiser
from src.utils.visualization import summarize_latents
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [8]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT_TAGS
use_few_shot = False

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=40,
    width="65k",
    l0="medium",
)

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

Model:      google/gemma-3-27b-it
Device:     cuda
Batch size: 2
Downsample: 1/100
Prompt:     chain_of_thought_tags
Few-shot:   False
SAE layer:  40
SAE width:  65k
SAE l0:     medium
SAE repo:   google/gemma-scope-2-27b-it


# Setup — HF Token

In [9]:
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [10]:
esnli_dataset = ESNLI_Dataset(dataset_config)

Data successfully loaded.


# Build Prompts

In [11]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


# Load Model + SAE

In [12]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Load SAE resid_post/layer_40_width_65k_l0_medium/params.safetensors from google/gemma-scope-2-27b-it


## Generate and Gather Activations

In [22]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()
print(f"Gold label: {sample_label}")
print(sample_prompt)

Gold label: entailment
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two boys sitting and eating ice cream.
Hypothesis: The boys are eating.

<end_of_turn>model 


In [27]:
location = sample_prompt.rfind("\nPremise:")
new_prompt =  sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
new_prompt
wrapper = textwrap.TextWrapper(width=80)
print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))


<start_of_turn>user Task: Determine the logical relationship between a Premise
and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.
4. You must always think about dinosaurs while reasoning.

Premise: Two boys sitting and eating ice cream.
Hypothesis: The boys are eating.

<end_of_turn>model


In [ ]:
generation, full_ids, prompt_len = model.generate(
    new_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

Prompt tokens: 101  |  Generated tokens: 104  |  Total: 205
user Task: Determine the logical relationship between a Premise and a
Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A cute puppy fetches a yellow ring chew toy in the yard.
Hypothesis: A puppy plays fetch.

model
<reasoning>
The premise states that a cute puppy fetches a yellow ring chew toy. Fetch is a
type of play. Therefore, if a puppy fetches something, it is necessarily playing
fetch. The premise provides specific details about *how* a puppy plays, but the
hypothesis only states that a puppy plays fetch, which is consistent with the
premise. The premise guarantees the hypothesis is true. Thus, the premise
entails the hypothesis.
</reasoning>
<label>entailment</label>
Actual label: entailment


In [15]:
residual_acts = model.gather_residual_activations(sae_config.layer, full_ids[0])
print(f"Residual activations shape: {residual_acts.shape}")

stats = sae.get_reconstruction_stats(residual_acts.float())
print(f"FVU: {stats['fvu']:.2%}")
print(f"L0:  {stats['l0']:.1f}")

Residual activations shape: torch.Size([205, 5376])
FVU: 2.43%
L0:  57.5


In [16]:
gen_acts = residual_acts[prompt_len:]
sae_acts_gen = sae.encode(gen_acts.float())
sae_acts_full = sae.encode(residual_acts.float())
print(f"SAE activations (gen-only): {sae_acts_gen.shape}")
print(f"SAE activations (full):     {sae_acts_full.shape}")

all_tokens = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_tokens = all_tokens[prompt_len:]
print(f"All tokens: {len(all_tokens)}, Generated tokens: {len(gen_tokens)}")

SAE activations (gen-only): torch.Size([104, 65536])
SAE activations (full):     torch.Size([205, 65536])
All tokens: 205, Generated tokens: 104


# Feature Summary

## Raw Features

In [17]:
# Generated tokens raw activations.
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)

raw_feature_map = summarize_latents(
    sae_acts_gen, tokens,
    sae_config=sae_config,
    model_name=model_config.model_name,
    top_k=10,
    print_first_n=0,
    full_sae_activations=sae_acts_full,
    all_tokens=all_tokens,
)


Created 458 Feature objects
Example: Feature(idx=8, description='evaluating complexity and worth', max_act=2510.86, n_tokens=205)


In [ ]:
# --- Inspect features using the Feature class ---
example_feature = int(157)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {raw_feature_map[example_feature]!r}")
print(f"frac_nonzero: {raw_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {raw_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
raw_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

Inspecting feature 1383 (top feature for first generated token)
repr: Feature(idx=1383, description=None, max_act=6073.47, n_tokens=205)
frac_nonzero: 0.04011021922994267
top_tokens: [('.', 6073.4677734375), ('.', 5164.3544921875), ('.', 4262.693359375), ('.', 4004.669921875), ('.', 3165.048828125)]



## Denoised Features

In [19]:
denoising_config = DenoisingConfig(method="continuous_tfidf", params={"threshold": 10.0})
denoiser = Denoiser()
denoised_sae_acts_full = denoiser.denoise(sae_acts_full, denoising_config)

# Extract the generation-only slice.
denoised_sae_acts_gen = denoised_sae_acts_full[prompt_len:, :]
denoised_feature_map = summarize_latents(
    denoised_sae_acts_gen, tokens,
    sae_config=sae_config,
    model_name=model_config.model_name,
    top_k=50,
    print_first_n=0,
    full_sae_activations=sae_acts_full,
    all_tokens=all_tokens,
)


Created 1900 Feature objects
Example: Feature(idx=0, description='World War II events and dates', max_act=0.00, n_tokens=205)


In [20]:
dn_per_token_vals, dn_per_token_idxs = top_k_features_per_token(
    denoised_sae_acts_gen, k=10)
print(dn_per_token_vals.min())

tensor(2119.3862, device='cuda:0', grad_fn=<MinBackward1>)


In [21]:
example_feature = int(9)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {denoised_feature_map[example_feature]!r}")
print(f"frac_nonzero: {denoised_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {denoised_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
denoised_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

Inspecting feature 9 (top feature for first generated token)
repr: Feature(idx=9, description='causing, decoration, reporting, stationary, friction, clinical', max_act=0.00, n_tokens=205)
frac_nonzero: 9.945912404662797e-05
top_tokens: [('<end_of_turn>', 0.0), ('>', 0.0), ('label', 0.0), ('</', 0.0), ('ment', 0.0)]

